In [1]:

# Cell 1 - Clone repo and install dependencies
!git clone https://github.com/JormayBusso/pixels-to-macros.git
%cd pixels-to-macros

!git fetch --all
!git reset --hard origin/dev

!pip install -q --upgrade pip
!pip install -q segmentation-models-pytorch timm datasets transformers albumentations opencv-python-headless safetensors tqdm Pillow
!grep -vE '^torch|^torchvision|coremltools' training/requirements.txt > /tmp/kaggle_reqs.txt
!pip install -q -r /tmp/kaggle_reqs.txt

import torch
print('CUDA:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
  print(i, torch.cuda.get_device_name(i))


Cloning into 'pixels-to-macros'...
remote: Enumerating objects: 16771, done.
remote: Counting objects: 100% (755/755), done.
remote: Compressing objects: 100% (645/645), done.
remote: Total 16771 (delta 239), reused 588 (delta 107), pack-reused 16016 (from 4)
Receiving objects: 100% (16771/16771), 1.55 GiB | 45.72 MiB/s, done.
Resolving deltas: 100% (1073/1073), done.
/kaggle/working/pixels-to-macros
Fetching origin
Updating files: 100% (14784/14784), done.
HEAD is now at 20fd96aa Kaggle Notebook | Pixels to macros Kaggle | first real run segformer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.9 MB/s eta 0:00:00
CUDA: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [2]:
# Cell 2 - Auto-Detect FoodSeg103 and Set Up
from pathlib import Path

# Use the exact path shown in your sidebar
DATA_DIR = Path('/kaggle/input/datasets/jormay/foodseg103-dataset/FoodSeg103')

# Verify it exists
assert DATA_DIR.exists(), f"Dataset not found at {DATA_DIR}. Check the sidebar!"

OUTPUT_DIR = Path('/kaggle/working/pixels-to-macros-segformer-103')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Success! Data found at: {DATA_DIR}")
print(f"🚀 Ready to start training!")

✅ Success! Data found at: /kaggle/input/datasets/jormay/foodseg103-dataset/FoodSeg103
🚀 Ready to start training!


In [3]:
# Cell 3 - Train SegFormer-B2 on FoodSeg103 (Fixed Arguments)
from pathlib import Path

MODEL       = 'nvidia/segformer-b2-finetuned-ade-512-512'
EPOCHS      = 80
BATCH_SIZE  = 8   
IMG_SIZE    = 512
LR          = 6e-5
NUM_WORKERS = 2
MULTIPLIER  = 3
VAL_EVERY   = 3

# I have updated the flag names to match your script's specific requirements
!python training/train.py \
  --data-dir       {DATA_DIR} \
  --output-dir     {OUTPUT_DIR} \
  --model-name     {MODEL} \
  --num-labels     104 \
  --epochs         {EPOCHS} \
  --batch-size     {BATCH_SIZE} \
  --img-size       {IMG_SIZE} \
  --lr             {LR} \
  --workers        {NUM_WORKERS} \
  --val-every      {VAL_EVERY} \
  --virtual-train-multiplier {MULTIPLIER} \
  --checkpoint-seconds 300

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Train samples: 4983 | Val samples: 2135
Virtual train multiplier: 3 (14949 augmented samples per epoch)
config.json: 6.88kB [00:00, 14.8MB/s]
pytorch_model.bin: 100%|█████████████████████| 110M/110M [00:02<00:00, 50.4MB/s]
Loading weights: 100%|█| 380/380 [00:00<00:00, 2342.99it/s, Materializing param=
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size(

In [4]:
# Cell 4 - Inspect and zip outputs (Make sure this matches OUTPUT_DIR)
!ls -lh /kaggle/working/pixels-to-macros-segformer-103
!tail -n 20 /kaggle/working/pixels-to-macros-segformer-103/metrics.json || true
!cd /kaggle/working && zip -r pixels-to-macros-segformer-103.zip pixels-to-macros-segformer-103

total 4.0K
-rw-r--r-- 1 root root 2.0K May 21 08:36 id2label.json
tail: cannot open '/kaggle/working/pixels-to-macros-segformer-103/metrics.json' for reading: No such file or directory
  adding: pixels-to-macros-segformer-103/ (stored 0%)
  adding: pixels-to-macros-segformer-103/id2label.json (deflated 59%)
